<a href="https://colab.research.google.com/github/joezein71/AIHC-5010-Winter-2026/blob/main/20260318_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving NLMCXR_reports.tgz to NLMCXR_reports.tgz
User uploaded file "NLMCXR_reports.tgz" with length 1112632 bytes


Once you've uploaded the `NLMCXR_reports.tgz` file, run the following cell to extract its contents.

In [28]:
# Extract the contents of the .tgz file
!tar -xzf NLMCXR_reports.tgz

In [ ]:
# Extract the contents of the .tgz file
!tar -xzf NLMCXR_reports.tgz

In [63]:
import os
import glob
import pandas as pd
import xml.etree.ElementTree as ET

# List all XML files in the 'ecgen-radiology' directory
xml_files = glob.glob('ecgen-radiology/*.xml')

data_rows = []

for file_path in xml_files:
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        # Initialize all extracted values to empty string. For Year/Month/Day, they'll become '' on failure.
        pmcid = os.path.basename(file_path).replace('.xml', '')
        uId = ''
        docSource = ''
        publisher = ''
        title = ''
        specialty = ''
        subset = ''
        comparison = ''
        indication = ''
        findings = ''
        impression = ''
        lastName = ''
        foreName = ''
        year = ''
        month = ''
        day = ''
        parentImage_id = ''
        caption = ''
        panel_single_content = ''

        # --- Extracting data based on XML structure assumptions ---
        # All findtext and get operations now explicitly default to ''

        # uId - Corrected path: it's an attribute of a top-level uId tag, not PMID under MedlineCitation
        uId_elem = root.find("./uId")
        if uId_elem is not None:
            uId = uId_elem.get('id') or ''

        # docSource - Corrected path: it's a top-level tag
        docSource = root.findtext("./docSource") or ''

        # publisher - Corrected path: it's a top-level tag
        publisher = root.findtext("./publisher") or ''

        # title - Can be top-level or under MedlineCitation/Article
        title = root.findtext("./title") or root.findtext(".//MedlineCitation/Article/ArticleTitle") or ''

        # specialty - Corrected path: it's a top-level tag
        specialty = root.findtext("./specialty") or ''

        # subset - Corrected path: it's a top-level tag
        subset = root.findtext("./subset") or ''

        # COMPARISON, INDICATION, FINDINGS, IMPRESSION from AbstractText with Label attribute - This path is generally correct
        for abstract_text_elem in root.findall(".//MedlineCitation/Article/Abstract/AbstractText"):
            label = abstract_text_elem.get('Label')
            text_content = abstract_text_elem.text or ''
            if label == 'COMPARISON':
                comparison = text_content
            elif label == 'INDICATION':
                indication = text_content
            elif label == 'FINDINGS':
                findings = text_content
            elif label == 'IMPRESSION':
                impression = text_content

        # LastName, ForeName (first author) - This path is generally correct
        author_elem = root.find(".//MedlineCitation/Article/AuthorList/Author")
        if author_elem is not None:
            lastName = author_elem.findtext('LastName') or ''
            foreName = author_elem.findtext('ForeName') or ''

        # Year, Month, Day from PubDate - Corrected path to PubDate within JournalIssue
        pub_date_elem = root.find(".//MedlineCitation/Article/Journal/JournalIssue/PubDate")
        if pub_date_elem is not None:
            year_text = pub_date_elem.findtext('Year')
            month_text = pub_date_elem.findtext('Month')
            day_text = pub_date_elem.findtext('Day')

            # Convert to integer if possible, otherwise use empty string
            try:
                year = str(int(year_text)) if year_text else ''
            except (ValueError, TypeError):
                year = ''
            try:
                month = str(int(month_text)) if month_text else ''
            except (ValueError, TypeError):
                month = ''
            try:
                day = str(int(day_text)) if day_text else ''
            except (ValueError, TypeError):
                day = ''

        # parentImage id, caption, panel content - Corrected paths
        parent_image_elem = root.find("./parentImage") # Top-level parentImage
        if parent_image_elem is not None:
            parentImage_id = parent_image_elem.get('id') or ''
            caption = parent_image_elem.findtext("./caption") or ''

            # Corrected: panel_single_content now extracts the text from the <url> tag inside <panel type="single">
            panel_single_elem = parent_image_elem.find("./panel[@type='single']")
            if panel_single_elem is not None:
                url_content = panel_single_elem.findtext("./url") or '' # Extract text from <url> child
                # Extract only what comes after 'extract/'
                if 'extract/' in url_content:
                    panel_single_content = url_content.split('extract/', 1)[1]
                else:
                    panel_single_content = url_content # Keep full URL if 'extract/' is not found

        data_rows.append({
            'pmcId': pmcid,
            'uId': uId,
            'docSource': docSource,
            'publisher': publisher,
            'title': title,
            'specialty': specialty,
            'subset': subset,
            'COMPARISON': comparison,
            'INDICATION': indication,
            'FINDINGS': findings,
            'IMPRESSION': impression,
            'LastName': lastName,
            'ForeName': foreName,
            'Year': year,
            'Month': month,
            'Day': day,
            'parentImage_id': parentImage_id,
            'caption': caption,
            'panel_type_single_content': panel_single_content
        })

    except ET.ParseError as e:
        print(f"Error parsing XML file {file_path}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while processing {file_path}: {e}")

# Create the DataFrame
df = pd.DataFrame(data_rows)

# Set pmcId as the index (rowname)
df = df.set_index('pmcId')

# Display the first 5 rows of the DataFrame
display(df.head())

,uId,docSource,publisher,title,specialty,subset,COMPARISON,INDICATION,FINDINGS,IMPRESSION,LastName,ForeName,Year,Month,Day,parentImage_id,caption,panel_type_single_content
pmcId,,,,,,,,,,,,,,,,,,
316,CXR316,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,,preop inguinal hernia history of laryngeal cancer,,Heart size normal. Lungs are clear. 5 mm right...,Kohli,Marc,2013,8,1,CXR316_IM-1487-1001,"PA and lateral chest XXXX, XXXX at XXXX with c...",CXR316_IM-1487-1001.jpg
1281,CXR1281,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,"XXXX, XXXX","Headache, XXXX, and congestion.",Normal heart size and mediastinal contours. Lu...,No acute cardiopulmonary process. .,Kohli,Marc,2013,8,1,CXR1281_IM-0188-2001,Xray Chest PA and Lateral,CXR1281_IM-0188-2001.jpg
3752,CXR3752,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,"XXXX, XXXX","XXXX, XXXX, EtOH",,1. Widened upper mediastinal silhouette. May r...,Kohli,Marc,2013,8,1,CXR3752_IM-1876-1001,"Chest 2 view XXXX, XXXX",CXR3752_IM-1876-1001.jpg
1719,CXR1719,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,CT of the chest with high-resolution from XXXX.,"XXXX-year-old male, XXXX for mass/infiltrate",The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary abnormality.,Kohli,Marc,2013,8,1,CXR1719_IM-0474-1001,2 view ( PA and lateral) chest radiograph date...,CXR1719_IM-0474-1001.jpg
1386,CXR1386,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,,Dyspnea,The cardiac silhouette and upper mediastinum a...,1. No acute pulmonary infiltrate or effusion. ...,Kohli,Marc,2013,8,1,CXR1386_IM-0246-1001,Frontal and lateral views of the chest were o...,CXR1386_IM-0246-1001.jpg


In [68]:
# Convert the pmcId index to numeric and then sort in ascending order
print("DEBUG: df['panel_type_single_content'].head() before copy:")
display(df['panel_type_single_content'].head())

df_sorted = df.copy()

print("DEBUG: df_sorted['panel_type_single_content'].head() after copy:")
display(df_sorted['panel_type_single_content'].head())

# Cleaning step: remove placeholders from text columns in df_sorted
import numpy as np
# Placeholders to remove, sorted by length in descending order to avoid partial matches
placeholders = ['XXXXX', 'XXXX', 'XXX', 'XX']

# Identify text columns (object dtype) within the df_sorted DataFrame
text_columns = df_sorted.select_dtypes(include='object').columns

# Apply replacements to each identified text column
for col in text_columns:
    # Ensure the column is of string type and handle NaNs for replacement
    # Fill NaN with empty string temporarily to allow str.replace
    temp_series = df_sorted[col].fillna('').astype(str)
    for p in placeholders:
        # Use regex=False for literal string replacement
        temp_series = temp_series.str.replace(p, '', regex=False)
    # Strip any leading/trailing whitespace that might remain after replacement
    temp_series = temp_series.str.strip()
    # Replace empty strings (or strings that became empty after replacement) with NaN
    # Note: If 'Year', 'Month', 'Day' are already '' from df creation, this step won't change them.
    df_sorted[col] = temp_series.replace('', np.nan)

print("DEBUG: df_sorted['panel_type_single_content'].head() after placeholder cleaning:")
display(df_sorted['panel_type_single_content'].head())

# Now, explicitly fill any remaining NaN (which could be from numeric columns if they weren't strings) with ''
# This ensures NO NaN values remain after all processing
df_sorted = df_sorted.fillna('')

print("DEBUG: df_sorted['panel_type_single_content'].head() after final fillna(''):")
display(df_sorted['panel_type_single_content'].head())

df_sorted.index = pd.to_numeric(df_sorted.index)
df_sorted = df_sorted.sort_index(ascending=True)

# Display the sorted DataFrame
display(df_sorted)

DEBUG: df['panel_type_single_content'].head() before copy:


,panel_type_single_content
pmcId,
316,CXR316_IM-1487-1001.jpg
1281,CXR1281_IM-0188-2001.jpg
3752,CXR3752_IM-1876-1001.jpg
1719,CXR1719_IM-0474-1001.jpg
1386,CXR1386_IM-0246-1001.jpg


DEBUG: df_sorted['panel_type_single_content'].head() after copy:


,panel_type_single_content
pmcId,
316,CXR316_IM-1487-1001.jpg
1281,CXR1281_IM-0188-2001.jpg
3752,CXR3752_IM-1876-1001.jpg
1719,CXR1719_IM-0474-1001.jpg
1386,CXR1386_IM-0246-1001.jpg


DEBUG: df_sorted['panel_type_single_content'].head() after placeholder cleaning:


,panel_type_single_content
pmcId,
316,CXR316_IM-1487-1001.jpg
1281,CXR1281_IM-0188-2001.jpg
3752,CXR3752_IM-1876-1001.jpg
1719,CXR1719_IM-0474-1001.jpg
1386,CXR1386_IM-0246-1001.jpg


DEBUG: df_sorted['panel_type_single_content'].head() after final fillna(''):


,panel_type_single_content
pmcId,
316,CXR316_IM-1487-1001.jpg
1281,CXR1281_IM-0188-2001.jpg
3752,CXR3752_IM-1876-1001.jpg
1719,CXR1719_IM-0474-1001.jpg
1386,CXR1386_IM-0246-1001.jpg


,uId,docSource,publisher,title,specialty,subset,COMPARISON,INDICATION,FINDINGS,IMPRESSION,LastName,ForeName,Year,Month,Day,parentImage_id,caption,panel_type_single_content
pmcId,,,,,,,,,,,,,,,,,,
1,CXR1,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,None.,Positive TB test,The cardiac silhouette and mediastinum size ar...,Normal chest x-.,Kohli,Marc,2013,8,1,CXR1_1_IM-0001-3001,Xray Chest PA and Lateral,CXR1_1_IM-0001-3001.jpg
2,CXR2,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,None.,Preop bariatric surgery.,Borderline cardiomegaly. Midline sternotomy . ...,No acute pulmonary findings.,Kohli,Marc,2013,8,1,CXR2_IM-0652-1001,"Chest, 2 views, frontal and lateral",CXR2_IM-0652-1001.jpg
3,CXR3,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,,"rib pain after a , steps this . Pain to R ba...",,"No displaced rib fractures, pneumothorax, or p...",Kohli,Marc,2013,8,1,CXR3_IM-1384-1001,Xray Chest PA and Lateral,CXR3_IM-1384-1001.jpg
4,CXR4,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,None available,-year-old with .,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,Kohli,Marc,2013,8,1,CXR4_IM-2050-1001,"PA and lateral views of the chest , at hours",CXR4_IM-2050-1001.jpg
5,CXR5,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,,Chest and nasal congestion.,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,Kohli,Marc,2013,8,1,CXR5_IM-2117-1003002,Xray Chest PA and Lateral,CXR5_IM-2117-1003002.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,CXR3995,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,", .","Nausea, vomiting x2 weeks. Dialysis patient.",The cardiomediastinal silhouette and pulmonary...,1. Interval resolution of bibasilar airspace d...,Kohli,Marc,2013,8,1,CXR3995_IM-2046-1001,Xray Chest PA and Lateral,CXR3995_IM-2046-1001.jpg
3996,CXR3996,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,None.,,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...,Kohli,Marc,2013,8,1,CXR3996_IM-2047-1001,Xray Chest PA and Lateral,CXR3996_IM-2047-1001.jpg
3997,CXR3997,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,None available.,-year-old male with positive PPD.,"Heart size within normal limits. Small, nodula...","No acute findings, no evidence for active TB.",Kohli,Marc,2013,8,1,CXR3997_IM-2048-1001,PA and lateral views of the chest.,CXR3997_IM-2048-1001.jpg


In [69]:
print("--- Info for 'panel_type_single_content' column ---")
df_sorted['panel_type_single_content'].info()

print("\n--- Descriptive Statistics for 'panel_type_single_content' ---")
display(df_sorted['panel_type_single_content'].describe(include='all'))

--- Info for 'panel_type_single_content' column ---
<class 'pandas.core.series.Series'>
Index: 3955 entries, 1 to 3999
Series name: panel_type_single_content
Non-Null Count  Dtype 
--------------  ----- 
3955 non-null   object
dtypes: object(1)
memory usage: 61.8+ KB

--- Descriptive Statistics for 'panel_type_single_content' ---


,panel_type_single_content
count,3955
unique,3852
top,
freq,104


This output will tell us definitively if the column is populated with data and provide details about its contents. If the 'Non-Null Count' is 3955, it means the column has data for every row.

In [70]:
selected_columns = [
    'uId', 'docSource', 'publisher', 'title', 'specialty', 'subset',
    'COMPARISON', 'LastName', 'ForeName', 'Year', 'Month', 'Day', 'caption'
]

print("--- Info for Selected Columns ---")
df_sorted[selected_columns].info()

print("\n--- Descriptive Statistics for Selected Columns ---")
display(df_sorted[selected_columns].describe(include='all'))

--- Info for Selected Columns ---
<class 'pandas.core.frame.DataFrame'>
Index: 3955 entries, 1 to 3999
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   uId         3955 non-null   object
 1   docSource   3955 non-null   object
 2   publisher   3955 non-null   object
 3   title       3955 non-null   object
 4   specialty   3955 non-null   object
 5   subset      3955 non-null   object
 6   COMPARISON  3955 non-null   object
 7   LastName    3955 non-null   object
 8   ForeName    3955 non-null   object
 9   Year        3955 non-null   object
 10  Month       3955 non-null   object
 11  Day         3955 non-null   object
 12  caption     3955 non-null   object
dtypes: object(13)
memory usage: 432.6+ KB

--- Descriptive Statistics for Selected Columns ---


,uId,docSource,publisher,title,specialty,subset,COMPARISON,LastName,ForeName,Year,Month,Day,caption
count,3955,3955,3955,3955,3955,3955,3955,3955,3955,3955,3955,3955,3955
unique,3955,1,1,1,1,1,390,1,1,1,1,1,607
top,CXR3999,CXR,Indiana University,Indiana University Chest X-ray Collection,pulmonary diseases,CXR,,Kohli,Marc,2013,8,1,Xray Chest PA and Lateral
freq,1,3955,3955,3955,3955,3955,913,3955,3955,3955,3955,3955,1236


In [93]:
# Install necessary libraries for text processing (if not already installed)
# This cell only needs to be run once.
%pip install nltk scikit-learn

In [94]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import pandas as pd

# Download necessary NLTK data (if not already downloaded)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True) # Open Multilingual Wordnet, often needed for WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text_for_clustering(text):
    text = str(text).lower()  # Convert to string and lowercase

    # Remove age-related terms (e.g., 'years old', 'year old', 'y/o')
    text = re.sub(r'\b(?:year|years|y/o)\b(?:\s*old)?', '', text)

    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation and numbers
    tokens = text.split()  # Tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Remove stop words and lemmatize
    return ' '.join(tokens)

# Apply preprocessing to the 'INDICATION' column
df_sorted['INDICATION_processed'] = df_sorted['INDICATION'].apply(preprocess_text_for_clustering)

print("Original INDICATION (first 5 rows):")
display(df_sorted['INDICATION'].head())

print("\nProcessed INDICATION (first 5 rows) without age terms:")
display(df_sorted['INDICATION_processed'].head())

Original INDICATION (first 5 rows):


,INDICATION
pmcId,
1,Positive TB test
2,Preop bariatric surgery.
3,"rib pain after a , steps this . Pain to R ba..."
4,-year-old with .
5,Chest and nasal congestion.



Processed INDICATION (first 5 rows) without age terms:


,INDICATION_processed
pmcId,
1,positive tb test
2,preop bariatric surgery
3,rib pain step pain r back r elbow r rib previo...
4,old
5,chest nasal congestion


In [111]:
import re

# Define keywords for each category (using preprocessed/lemmatized forms where appropriate)
categories = {
    'Positive TB Test': ['positive tb test', 'tb test positive', 'tb positive', 'ppd', 'tb'],
    'Dyspnea': ['dyspnea', 'breath', 'shortness', 'asthma', 'sob', 'hypoxia'],
    'Chest Pain or Rib Pain': ['chest pain', 'rib pain', 'pain'],
    'Respiratory Issues & Infection': ['chest congestion', 'congestion', 'rule out infection', 'infection rule out', 'rule infection', 'pneumonia', 'copd', 'productive'],
    'Cardiac': ['heart disease', 'cardiac', 'cardiomegaly', 'coronary', 'chf', 'mi', 'hypertension', 'syncope'],
    'Malignancy': ['cancer', 'neoplasm'],
    'Transplant': ['transplant'],
    'Preop / Evaluation': ['preop', 'preoperative', 'pre op', 'pre operative', 'evaluation', 'workup']
}

def assign_custom_category(processed_text):
    processed_text = str(processed_text) # Ensure it's a string
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in processed_text:
                return category
    return 'Other/Uncategorized' # Default category if no match

# Assign custom categories
df_sorted['custom_indication_category'] = df_sorted['INDICATION_processed'].apply(assign_custom_category)

print("First 10 rows with new custom categories:")
display(df_sorted[['INDICATION', 'INDICATION_processed', 'custom_indication_category']].head(10))

print("\nDistribution of custom categories:")
display(df_sorted['custom_indication_category'].value_counts())

First 10 rows with new custom categories:


,INDICATION,INDICATION_processed,custom_indication_category
pmcId,,,
1,Positive TB test,positive tb test,Positive TB Test
2,Preop bariatric surgery.,preop bariatric surgery,Preop / Evaluation
3,"rib pain after a , steps this . Pain to R ba...",rib pain step pain r back r elbow r rib previo...,Chest Pain or Rib Pain
4,-year-old with .,old,Other/Uncategorized
5,Chest and nasal congestion.,chest nasal congestion,Respiratory Issues & Infection
6,Evaluate for infection,evaluate infection,Other/Uncategorized
7,Preop lumbar surgery,preop lumbar surgery,Preop / Evaluation
8,-year-old with on . Dyspnea. History of mitra...,old dyspnea history mitral valve prolapse,Dyspnea
9,Chest pain today. History of stent placement 7...,chest pain today history stent placement ago,Chest Pain or Rib Pain



Distribution of custom categories:


,count
custom_indication_category,
Other/Uncategorized,1398
Chest Pain or Rib Pain,1073
Dyspnea,780
Preop / Evaluation,168
Cardiac,158
Respiratory Issues & Infection,137
Positive TB Test,93
Malignancy,90
Transplant,58
